In [1]:
import torch
import torch.optim as optim
import torch.nn as nn
from torchvision import transforms, datasets
from torch.utils.data import DataLoader, random_split
from Models import CNN_1

import matplotlib.pyplot as plt

In [2]:
batch_size_=32
num_epochs = 50

In [3]:
train_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.RandomHorizontalFlip(0.5),
    transforms.RandomRotation(30),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

val_test_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

In [4]:

full_dataset = datasets.ImageFolder("./DATASET/FULL", transform=transforms.ToTensor())

total_size = len(full_dataset)
train_size = int(0.7 * total_size)
val_size = int(0.2 * total_size)
test_size = total_size - train_size - val_size

train_dataset, val_dataset, test_dataset = random_split(full_dataset, [train_size, val_size, test_size])

In [5]:
from torch.utils.data import Subset

class SubsetWithTransform(Subset):
    def __init__(self, subset, transform=None):
        super().__init__(subset.dataset, subset.indices)
        self.transform = transform

    def __getitem__(self, idx):
        x, y = super().__getitem__(idx)
        if self.transform:
            x = self.transform(x)
        return x, y

train_loader= SubsetWithTransform(train_dataset, transform=train_transform)
val_loader = SubsetWithTransform(val_dataset, transform=val_test_transform)
test_loader = SubsetWithTransform(test_dataset, transform=val_test_transform)

In [6]:
train_dataset = DataLoader(train_loader, batch_size=batch_size_, shuffle=True)
val_dataset = DataLoader(val_loader, batch_size=batch_size_, shuffle=False)
test_dataset  = DataLoader(test_loader, batch_size=batch_size_, shuffle=False)

In [7]:

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model1 = CNN_1().to(device)
criterion = nn.BCELoss()
optimizer = optim.Adam(model1.parameters(), lr=0.001)


In [8]:
train_losses = []
val_losses = []
train_accurate = []
val_accuarate = []

for epoch in range(num_epochs):
    model1.train()

    total_loss = 0
    correct_train = 0
    total_train = 0

    for images, labels in train_dataset:
        images = images.to(device)
        labels = labels.float().unsqueeze(1).to(device)

        outputs = model1(images)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        preds = (outputs > 0.5).int()
        correct_train += (preds == labels.int()).sum().item()
        total_train += labels.size(0)

    train_loss = total_loss / len(train_dataset)
    train_accuracy = correct_train / total_train

    train_losses.append(train_loss)
    train_accurate.append(train_accuracy)

    model1.eval()
    val_loss_total = 0
    correct_val = 0
    total_val = 0

    with torch.no_grad():
        for images, labels in val_dataset:
            images = images.to(device)
            labels = labels.float().unsqueeze(1).to(device)

            outputs = model1(images)
            loss = criterion(outputs, labels)

            val_loss_total += loss.item()

            preds = (outputs > 0.5).int()
            correct_val += (preds == labels.int()).sum().item()
            total_val += labels.size(0)

    val_loss = val_loss_total / len(val_dataset)
    val_accuracy = correct_val / total_val

    val_losses.append(val_loss)
    val_accuarate.append(val_accuracy)

    print(f"Epoch {epoch+1}: "
          f"Train Loss={train_loss:.4f}, Train Acc={train_accuracy:.4f} | "
          f"Val Loss={val_loss:.4f}, Val Acc={val_accuracy:.4f}")

RuntimeError: stack expects each tensor to be equal size, but got [3, 193, 261] at entry 0 and [3, 183, 276] at entry 1

In [ ]:
def plot_graphs(x1, x2, string):
    plt.plot(x1, label='train_' + string)
    plt.plot(x2, label='val_' + string)
    plt.xlabel("Epochs")
    plt.ylabel(string)
    plt.title(string + " over epochs")
    plt.legend()
    plt.show()

In [ ]:
 plot_graphs(train_losses, val_losses, "Loss")

In [ ]:
plot_graphs(train_accurate,val_accuarate,"Accuracy")

In [ ]:
model1.eval()

with torch.no_grad():
    for images, labels in Test_dataset:
        images = images.to(device)

        outputs = model1(images)
        preds = (outputs > 0.5).int()

        print(preds)
        break